In [30]:
import os
import pickle
import numpy as np
import pandas as pd

from dotenv import load_dotenv
from itertools import product
from scipy.stats import loguniform, uniform

from sklearn.base import clone
from sklearn.linear_model import SGDClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import TimeSeriesSplit, ParameterSampler
from sklearn.metrics import average_precision_score

from sqlalchemy import create_engine, text

from customer_analytics.churn_prediction.database import ClassifierDataIterator

In [31]:
load_dotenv()
engine = create_engine(os.getenv("DATABASE_URL"))
seed = 123
n_iter = 30

In [32]:
with open("../data/classifier_techniques.pkl", "rb") as f:
    classifier_techniques = pickle.load(f)

In [33]:
classifier_techniques["logistic_regression"]["params_distribution"] = {
    "alpha": loguniform(1e-6, 1e-2),
    "l1_ratio": uniform(0, 1),
    "learning_rate": ["optimal", "adaptive", "invscaling"],
    "eta0": loguniform(1e-4, 1e-1),
}

classifier_techniques["support_vector_machine"]["params_distribution"] = {
    "alpha": loguniform(1e-6, 1e-2),
    "learning_rate": ["optimal", "adaptive", "invscaling"],
    "eta0": loguniform(1e-4, 1e-1),
}

classifier_techniques["mlp_classifier"]["params_distribution"] = {
    "hidden_layer_sizes": [
        arch
        for n_layers in range(1, 4)
        for arch in product([32, 64, 128], repeat=n_layers)
    ],
    "activation": ["logistic", "tanh", "relu"],
    "alpha": loguniform(1e-6, 1e-2),
    "learning_rate_init": loguniform(1e-5, 1e-2),
}

classifier_techniques["naive_bayes"]["params_distribution"] = {
    "var_smoothing": loguniform(1e-12, 1e-6)
}

classifier_techniques["xgboost"]["params_distribution"] = {
    "n_estimators": [100, 200, 300, 500, 750, 1000],
    "max_depth": [3, 4, 5, 6, 8, 10],
    "learning_rate": loguniform(0.01, 0.3),
    "subsample": uniform(0.6, 0.4),
    "colsample_bytree": uniform(0.6, 0.4),
    "min_child_weight": loguniform(1, 20),
    "gamma": uniform(0, 5),
    "reg_alpha": loguniform(1e-5, 10),
    "reg_lambda": loguniform(1e-3, 100)
}

In [34]:
dates = pd.read_sql_query(
    text("""
        SELECT DISTINCT cutoff_date
        FROM churn_model_dataset
        WHERE dataset_split = 'train'
          AND age IS NOT NULL
        ORDER BY cutoff_date
    """),
    engine
)["cutoff_date"].to_numpy()

tscv = TimeSeriesSplit(
    n_splits=4,
    test_size=int(len(dates) * 0.2)
)

folds = []

for train_idx, val_idx in tscv.split(dates):
    folds.append({
        "validation_start": dates[val_idx[0]],
        "validation_end": dates[val_idx[-1]] + np.timedelta64(1, "D")
    })

In [ ]:
for classifier_name, technique in classifier_techniques.items():

    candidates = ParameterSampler(
        technique["params_distribution"],
        n_iter=n_iter,
        random_state=seed
    )

    cv_results = []

    for candidate_idx, params in enumerate(candidates):

        print(
            f"{classifier_name}: "
            f"candidate {candidate_idx + 1}/{n_iter}"
        )

        fold_scores = []

        for fold_idx, fold in enumerate(folds):

            model = clone(technique["method"])
            model.set_params(**params)

            train_iterator = ClassifierDataIterator(
                engine=engine,
                split="train",
                batch_size=10_000,
                end_date=fold["validation_start"]
            )

            validation_iterator = ClassifierDataIterator(
                engine=engine,
                split="train",
                batch_size=10_000,
                start_date=fold["validation_start"],
                end_date=fold["validation_end"]
            )

            first_batch = True

            for X_batch, y_batch in train_iterator.get_batches():

                if first_batch:
                    model.partial_fit(
                        X_batch,
                        y_batch,
                        classes=np.array([0, 1])
                    )
                    first_batch = False

                else:
                    model.partial_fit(X_batch, y_batch)

            if first_batch:
                raise ValueError(
                    f"No training observations for {classifier_name}"
                )

            y_true = []
            y_scores = []

            for X_batch, y_batch in validation_iterator.get_batches():

                y_true.extend(y_batch)

                if hasattr(model, "predict_proba"):
                    scores = model.predict_proba(X_batch)[:, 1]
                else:
                    scores = model.decision_function(X_batch)

                y_scores.extend(scores)

            y_true = np.asarray(y_true)
            y_scores = np.asarray(y_scores)

            if len(y_true) == 0 or len(np.unique(y_true)) < 2:
                raise ValueError(
                    f"Invalid validation fold for {classifier_name}, "
                    f"fold {fold_idx + 1}"
                )

            fold_score = average_precision_score(
                y_true,
                y_scores
            )

            fold_scores.append(fold_score)

            print(
                f"  Fold {fold_idx + 1}: "
                f"Average Precision = {fold_score:.4f}"
            )

        cv_results.append({
            "params": params,
            "mean_average_precision": np.mean(fold_scores),
            "std_average_precision": np.std(fold_scores),
            "fold_scores": fold_scores
        })

    # Rank candidates by mean average precision
    cv_results.sort(
        key=lambda x: x["mean_average_precision"],
        reverse=True
    )

    # Store results in classifier_techniques
    technique["cv_results"] = cv_results
    technique["best_params"] = cv_results[0]["params"]
    technique["best_score"] = cv_results[0][
        "mean_average_precision"
    ]

logistic_regression: candidate 1/30
  Fold 1: Average Precision = 0.9155
  Fold 2: Average Precision = 0.9135


KeyboardInterrupt: 

In [ ]:
with open("../data/classifier_techniques.pkl", "wb") as file:
    pickle.dump(classifier_techniques, file)